# Differentiable Quantum Architecture Search

## Overview

This tutorial demonstrates how to utilize the advanced computational features provided by TensorCircuit such as ``jit`` and ``vmap`` to super efficiently simulate the differentiable quantum architecture search (DQAS) algorithm, where an ensemble of quantum circuits with different structures can be compiled to simulate at the same time.

[WIP note]

## Setup

In [1]:
import numpy as np
import tensorcircuit as tc
import tensorflow as tf

In [2]:
K = tc.set_backend("tensorflow")
ctype, rtype = tc.set_dtype("complex128")

## Problem Description

The task is to find the state preparation circuit for GHZ state $\vert \text{GHZ}_N\rangle = \frac{1}{\sqrt{2}}\left(\vert 0^N\rangle +\vert 1^N\rangle \right)$. We prepare a gate pool with rx0, rx1, ry0, ry1, rz0, rz1, cnot01, cnot10 for the $N=2$ demo. Amongst the eight gates, six are parameterized.

In [3]:
def rx0(theta):
    return K.kron(
        K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._x_matrix, K.eye(2)
    )


def rx1(theta):
    return K.kron(
        K.eye(2), K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._x_matrix
    )


def ry0(theta):
    return K.kron(
        K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._y_matrix, K.eye(2)
    )


def ry1(theta):
    return K.kron(
        K.eye(2), K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._y_matrix
    )


def rz0(theta):
    return K.kron(
        K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._z_matrix, K.eye(2)
    )


def rz1(theta):
    return K.kron(
        K.eye(2), K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._z_matrix
    )


def cnot01():
    return K.cast(K.convert_to_tensor(tc.gates._cnot_matrix), ctype)


def cnot10():
    return K.cast(
        K.convert_to_tensor(
            np.array([[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0]])
        ),
        ctype,
    )


ops_repr = ["rx0", "rx1", "ry0", "ry1", "rz0", "rz1", "cnot01", "cnot10"]

In [4]:
n, p, ch = 2, 3, 8
# number of qubits, number of layers, size of operation pool

target = tc.array_to_tensor(np.array([1, 0, 0, 1.0]) / np.sqrt(2.0))
# target wavefunction, we here use GHZ2 state as the objective target


def ansatz(params, structures):
    c = tc.Circuit(n)
    params = K.cast(params, ctype)
    structures = K.cast(structures, ctype)
    for i in range(p):
        c.any(
            0,
            1,
            unitary=structures[i, 0] * rx0(params[i, 0])
            + structures[i, 1] * rx1(params[i, 1])
            + structures[i, 2] * ry0(params[i, 2])
            + structures[i, 3] * ry1(params[i, 3])
            + structures[i, 4] * rz0(params[i, 4])
            + structures[i, 5] * rz1(params[i, 5])
            + structures[i, 6] * cnot01()
            + structures[i, 7] * cnot10(),
        )
    s = c.state()
    loss = K.sum(K.abs(target - s))
    return loss


vag1 = K.jit(K.vvag(ansatz, argnums=0, vectorized_argnums=1))

## Probability Ensemble Approach

This approach is more practical and experimental relevant and is the same algorithm described in Ref.1, though we here use advanced vmap to accelerate the simulation of circuits with different structures.

In [5]:
def sampling_from_structure(structures, batch=1):
    prob = K.softmax(K.real(structures), axis=-1)
    return np.array([np.random.choice(ch, p=K.numpy(prob[i])) for i in range(p)])


@K.jit
def best_from_structure(structures):
    return K.argmax(structures, axis=-1)


@K.jit
def nmf_gradient(structures, oh):
    """
    compute the Monte Carlo gradient with respect of naive mean-field probabilistic model
    """
    choice = K.argmax(oh, axis=-1)
    prob = K.softmax(K.real(structures), axis=-1)
    indices = K.transpose(K.stack([K.cast(tf.range(p), "int64"), choice]))
    prob = tf.gather_nd(prob, indices)
    prob = K.reshape(prob, [-1, 1])
    prob = K.tile(prob, [1, ch])

    return tf.tensor_scatter_nd_add(
        tf.cast(-prob, dtype=ctype),
        indices,
        tf.ones([p], dtype=ctype),
    )


nmf_gradient_vmap = K.vmap(nmf_gradient, vectorized_argnums=1)

In [6]:
verbose = False
epochs = 400
batch = 256
lr = tf.keras.optimizers.schedules.ExponentialDecay(0.06, 100, 0.5)
structure_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(0.12))
network_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(lr))
nnp = K.implicit_randn(stddev=0.02, shape=[p, 6], dtype=rtype)
stp = K.implicit_randn(stddev=0.02, shape=[p, 8], dtype=rtype)
avcost1 = 0
for epoch in range(epochs):  # iteration to update strcuture param
    avcost2 = avcost1
    costl = []
    batched_stuctures = K.onehot(
        np.stack([sampling_from_structure(stp) for _ in range(batch)]), num=8
    )
    infd, gnnp = vag1(nnp, batched_stuctures)
    gs = nmf_gradient_vmap(stp, batched_stuctures)  # \nabla lnp
    gstp = [K.cast((infd[i] - avcost2), ctype) * gs[i] for i in range(infd.shape[0])]
    gstp = K.real(K.sum(gstp, axis=0) / infd.shape[0])
    avcost1 = K.sum(infd) / infd.shape[0]
    nnp = network_opt.update(gnnp, nnp)
    stp = structure_opt.update(gstp, stp)

    if epoch % 40 == 0 or epoch == epochs - 1:
        print("----------epoch %s-----------" % epoch)
        print(
            "batched average loss: ",
            np.mean(avcost1),
        )

        if verbose:
            print(
                "strcuture parameter: \n",
                stp.numpy(),
                "\n network parameter: \n",
                nnp.numpy(),
            )

        cand_preset = best_from_structure(stp)
        print("best candidates so far:", [ops_repr[i] for i in cand_preset])
        print(
            "corresponding weights for each gate:",
            [K.numpy(nnp[j, i]) if i < 6 else 0.0 for j, i in enumerate(cand_preset)],
        )

----------epoch 0-----------
batched average loss:  1.0149079390928712
best candidates so far: ['cnot01', 'rx0', 'rx1']
corresponding weights for each gate: [0.0, np.float64(-0.038394295344198574), np.float64(0.04230785330885818)]


----------epoch 40-----------
batched average loss:  0.9905816893778643
best candidates so far: ['cnot10', 'rz1', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(0.002168744963904897), 0.0]


----------epoch 80-----------
batched average loss:  0.012791440476505156
best candidates so far: ['cnot10', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7707101909239031), 0.0]


----------epoch 120-----------
batched average loss:  0.014319409802050905
best candidates so far: ['cnot10', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7760508621727651), 0.0]


----------epoch 160-----------
batched average loss:  0.005774592367694731
best candidates so far: ['cnot10', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7792065565668621), 0.0]


...

----------epoch 240-----------
batched average loss:  0.003395389408270777
best candidates so far: ['cnot10', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7818729681866823), 0.0]


----------epoch 280-----------
batched average loss:  0.001126982639521934
best candidates so far: ['cnot10', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7855536262495508), 0.0]


----------epoch 320-----------
batched average loss:  0.00015998827740693056
best candidates so far: ['cnot10', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7847162255707778), 0.0]


----------epoch 360-----------
batched average loss:  0.0001513488500263227
best candidates so far: ['cnot10', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7848986211228433), 0.0]


----------epoch 399-----------
batched average loss:  0.003939997443752287
best candidates so far: ['cnot10', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7861293606583808), 0.0]


## Directly Optimize the Structure Parameters

Since we are using numerical simulation anyway, we can directly optimize the structure parameter and omit whether the super circuit is unitary or not, this approach can be faster and more reliable for some scenarios.

In [7]:
def ansatz2(params, structures):
    c = tc.Circuit(n)
    params = K.cast(params, ctype)
    structures = K.softmax(structures, axis=-1)
    structures = K.cast(structures, ctype)
    for i in range(p):
        c.any(
            0,
            1,
            unitary=structures[i, 0] * rx0(params[i, 0])
            + structures[i, 1] * rx1(params[i, 1])
            + structures[i, 2] * ry0(params[i, 2])
            + structures[i, 3] * ry1(params[i, 3])
            + structures[i, 4] * rz0(params[i, 4])
            + structures[i, 5] * rz1(params[i, 5])
            + structures[i, 6] * cnot01()
            + structures[i, 7] * cnot10(),
        )
    s = c.state()
    s /= K.norm(s)
    loss = K.sum(K.abs(target - s))
    return loss


vag2 = K.jit(K.value_and_grad(ansatz2, argnums=(0, 1)))

In [8]:
verbose = True
epochs = 700
lr = tf.keras.optimizers.schedules.ExponentialDecay(0.05, 200, 0.5)
structure_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(0.04))
network_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(lr))
nnp = K.implicit_randn(stddev=0.02, shape=[p, 6], dtype=rtype)
stp = K.implicit_randn(stddev=0.02, shape=[p, 8], dtype=rtype)
for epoch in range(epochs):
    infd, (gnnp, gstp) = vag2(nnp, stp)

    nnp = network_opt.update(gnnp, nnp)
    stp = structure_opt.update(gstp, stp)
    if epoch % 70 == 0 or epoch == epochs - 1:
        print("----------epoch %s-----------" % epoch)
        print(
            "batched average loss: ",
            np.mean(infd),
        )
        if verbose:
            print(
                "strcuture parameter: \n",
                stp.numpy(),
                "\n network parameter: \n",
                nnp.numpy(),
            )

        cand_preset = best_from_structure(stp)
        print("best candidates so far:", [ops_repr[i] for i in cand_preset])
        print(
            "corresponding weights for each gate:",
            [K.numpy(nnp[j, i]) if i < 6 else 0.0 for j, i in enumerate(cand_preset)],
        )

----------epoch 0-----------
batched average loss:  1.010720632517599
strcuture parameter: 
 [[ 0.03214832 -0.0320775   0.02739976  0.05935448  0.02540355  0.05371278
   0.02400283  0.02890031]
 [-0.04258084  0.04329034 -0.00383521  0.03037485  0.04368983  0.02515855
   0.06912776  0.04965446]
 [ 0.0168632  -0.0299822   0.02514552 -0.04125121  0.05772422  0.02461286
   0.06444225 -0.05064111]] 
 network parameter: 
 [[ 0.0501497   0.03282115 -0.05105747  0.07116612  0.05195183  0.02769577]
 [ 0.00303168  0.04752419 -0.0304752   0.06967789 -0.03375994  0.06062084]
 [ 0.05102347  0.03209849 -0.0595332   0.00301597 -0.05449674  0.05215252]]
best candidates so far: ['ry1', 'cnot01', 'cnot01']
corresponding weights for each gate: [np.float64(0.07116612229072498), 0.0, 0.0]


----------epoch 70-----------
batched average loss:  0.6376604031609525
strcuture parameter: 
 [[-1.36023746 -1.59290465  1.29905395 -0.82748593 -0.52927769 -0.5001411
  -0.5009181  -0.49601913]
 [-0.86779795 -1.13893206  0.06451843 -0.58200512 -0.80741698 -0.79900022
   0.88326954  1.94830163]
 [-1.87011733 -2.03741249  0.93572937 -0.51756493 -1.10490015 -1.12573768
   1.62707197 -1.30962241]] 
 network parameter: 
 [[-1.17513918e-02 -7.92546969e-03 -1.17533575e+00 -5.12253151e-02
   8.62800743e-02  1.00414599e-02]
 [-2.72186753e-02 -3.44780713e-03 -4.32455369e-01  2.55599944e-01
  -4.80141937e-02 -1.74882052e-04]
 [-1.46157790e-02  1.45161161e-02  1.20690438e+00  4.94757043e-01
  -2.08760140e-01 -4.64604973e-02]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.1753357538303417), 0.0, 0.0]


----------epoch 140-----------
batched average loss:  0.005415002608940208
strcuture parameter: 
 [[-1.6877884  -1.92517314  1.67415662 -1.28249833 -0.87656248 -0.84883964
  -0.82578313 -0.8208842 ]
 [-0.32023879 -0.66825832  0.43847607 -0.6063058  -0.3089614  -0.39763217
   0.22348193  2.90371747]
 [-2.47636906 -2.6414169   1.03425924 -1.09967921 -1.76194271 -1.7434864
   2.26139679 -1.96729729]] 
 network parameter: 
 [[-0.01765424 -0.01288028 -1.38519416  0.0074377   0.0843547   0.00430166]
 [-0.03143036 -0.00449565 -0.59472769  0.2962399  -0.05754176 -0.03663302]
 [-0.02635261  0.02965428  1.44856863  0.40791152 -0.20017575 -0.14844691]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.3851941553587614), 0.0, 0.0]


----------epoch 210-----------
batched average loss:  0.014742149189165388
strcuture parameter: 
 [[-1.6861452  -1.9262005   1.67648304 -1.29576663 -0.88155241 -0.85338399
  -0.82317684 -0.81827791]
 [-0.27478072 -0.66914128  0.38361075 -0.62907435 -0.28409667 -0.38977183
   0.18087223  3.01489872]
 [-2.49085027 -2.65589445  1.00197743 -1.13070903 -1.78102161 -1.76110271
   2.30058941 -1.97944149]] 
 network parameter: 
 [[-0.01907793 -0.00300235 -1.36078523  0.00411351  0.07144835 -0.00305581]
 [-0.0336473   0.00885847 -0.56992524  0.3022262  -0.0531588  -0.04155323]
 [-0.0268694   0.03167755  1.44389629  0.40000706 -0.20122162 -0.14149867]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.3607852284148154), 0.0, 0.0]


----------epoch 280-----------
batched average loss:  0.002966539872295006
strcuture parameter: 
 [[-1.68576619 -1.92733137  1.67804895 -1.30017453 -0.88629475 -0.85834345
  -0.82113713 -0.81623821]
 [-0.31033939 -0.72301021  0.36041166 -0.68154182 -0.32418601 -0.45577419
   0.19176055  3.05189182]
 [-2.49003891 -2.6558173   0.99419795 -1.15209577 -1.78610266 -1.7662965
   2.31079862 -1.97587693]] 
 network parameter: 
 [[-1.82590567e-02 -6.95271343e-03 -1.35269714e+00 -1.76560597e-02
   7.38375172e-02  5.31290384e-04]
 [-3.01693772e-02  2.08346972e-03 -5.60905697e-01  2.82479298e-01
  -4.96206788e-02 -3.57659637e-02]
 [-2.61958738e-02  2.98503390e-02  1.43641938e+00  3.82792226e-01
  -2.00573457e-01 -1.30213949e-01]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.3526971397641652), 0.0, 0.0]


...

----------epoch 420-----------
batched average loss:  0.006825582114435391
strcuture parameter: 
 [[-1.6941456  -1.93724286  1.68792251 -1.31662761 -0.89730709 -0.8694384
  -0.82808628 -0.82318736]
 [-0.28961047 -0.74058378  0.29804131 -0.73766264 -0.32769246 -0.45336073
   0.18910798  3.13619183]
 [-2.47184345 -2.63910835  0.95664746 -1.1693814  -1.77444112 -1.75327225
   2.34673586 -1.95893177]] 
 network parameter: 
 [[-2.05967012e-02 -8.73155007e-03 -1.33751719e+00 -4.02945102e-02
   7.38457606e-02  2.14420563e-03]
 [-2.93059484e-02 -9.80577049e-04 -5.42361494e-01  2.72433027e-01
  -4.16664759e-02 -3.52149962e-02]
 [-2.58013576e-02  2.76609285e-02  1.41875362e+00  3.66107674e-01
  -1.97954660e-01 -1.17121325e-01]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.337517192068601), 0.0, 0.0]


----------epoch 490-----------
batched average loss:  0.013623399755292056
strcuture parameter: 
 [[-1.70006926 -1.94281916  1.69351822 -1.32535898 -0.90319021 -0.87489238
  -0.83256539 -0.82766647]
 [-0.26861844 -0.74287136  0.27885004 -0.73602101 -0.33039778 -0.44887653
   0.15906925  3.18341837]
 [-2.47920732 -2.64624958  0.94027132 -1.17575988 -1.78265838 -1.7605194
   2.36487837 -1.96683176]] 
 network parameter: 
 [[-0.01532878 -0.00613218 -1.33360321 -0.04105805  0.06960364 -0.00196048]
 [-0.02874709  0.00260475 -0.53769807  0.27527362 -0.05186812 -0.03130756]
 [-0.02657409  0.03059073  1.41576708  0.36630394 -0.19803571 -0.12453668]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.3336032111825327), 0.0, 0.0]


----------epoch 560-----------
batched average loss:  0.015478248613106912
strcuture parameter: 
 [[-1.68880735 -1.93181534  1.68244264 -1.31551895 -0.89204811 -0.8638329
  -0.82117705 -0.81627812]
 [-0.23443387 -0.72841177  0.2365013  -0.76313691 -0.30479435 -0.41714282
   0.16082371  3.22924122]
 [-2.47984933 -2.64728589  0.94204437 -1.19379857 -1.7858053  -1.76258259
   2.36509792 -1.96469023]] 
 network parameter: 
 [[-1.56479356e-02 -6.72039567e-03 -1.32575944e+00 -4.77314775e-02
   6.99568247e-02 -7.34583579e-04]
 [-2.82210616e-02  2.55977559e-03 -5.29582557e-01  2.70075668e-01
  -4.97719851e-02 -3.11413801e-02]
 [-2.67045951e-02  3.08076613e-02  1.41401204e+00  3.60039722e-01
  -1.95698416e-01 -1.26077635e-01]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.3257594384303728), 0.0, 0.0]


----------epoch 630-----------
batched average loss:  0.011927057567909535
strcuture parameter: 
 [[-1.69249841 -1.9355087   1.68628654 -1.32009273 -0.89605684 -0.86751377
  -0.82485084 -0.81995192]
 [-0.20555873 -0.70634433  0.21063173 -0.77405128 -0.27513036 -0.39128084
   0.15063109  3.26422603]
 [-2.48547983 -2.65331075  0.93082062 -1.21106371 -1.78837065 -1.76949581
   2.37852059 -1.96876016]] 
 network parameter: 
 [[-1.51264282e-02 -6.63218399e-03 -1.32249131e+00 -5.20409790e-02
   6.78638945e-02 -2.77112467e-03]
 [-2.90347621e-02  8.47142907e-04 -5.25745461e-01  2.68426773e-01
  -5.20063586e-02 -3.18315449e-02]
 [-2.64407621e-02  2.98609776e-02  1.41215163e+00  3.56196745e-01
  -2.00112333e-01 -1.21685352e-01]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.322491308434586), 0.0, 0.0]


----------epoch 699-----------
batched average loss:  0.008211551172844417
strcuture parameter: 
 [[-1.70091689 -1.94420589  1.69473203 -1.32804936 -0.90452406 -0.87622873
  -0.83325977 -0.82836085]
 [-0.19501115 -0.70327154  0.19498828 -0.79383108 -0.2639047  -0.38106542
   0.14822215  3.28593978]
 [-2.49131892 -2.65935777  0.91880057 -1.23363116 -1.79502934 -1.7756798
   2.3932033  -1.97238143]] 
 network parameter: 
 [[-1.56983635e-02 -7.10114995e-03 -1.32105719e+00 -5.60844899e-02
   6.94539849e-02 -1.24711837e-03]
 [-2.82893419e-02  4.71919772e-04 -5.23884755e-01  2.65679294e-01
  -4.98091963e-02 -3.12930005e-02]
 [-2.63684623e-02  2.97217530e-02  1.41097355e+00  3.52070340e-01
  -1.99013612e-01 -1.21667537e-01]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.3210571927012826), 0.0, 0.0]


## Final Fine-tune

For the obtained circuit layout we can further adjust the circuit weights to make the objective more close to zero.

In [9]:
chosen_structure = K.onehot(np.array([2, 4, 6]), num=8)
chosen_structure = K.reshape(chosen_structure, [1, p, ch])
chosen_structure

<tf.Tensor: shape=(1, 3, 8), dtype=float32, numpy=
array([[[0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0.]]], dtype=float32)>

In [10]:
network_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(1e-3))
nnp = K.implicit_randn(stddev=0.02, shape=[p, 6], dtype=rtype)
verbose = True
epochs = 600
for epoch in range(epochs):
    infd, gnnp = vag1(nnp, chosen_structure)
    nnp = network_opt.update(gnnp, nnp)
    if epoch % 60 == 0 or epoch == epochs - 1:
        print(epoch, "loss: ", K.numpy(infd[0]))

0 loss:  1.008930882702804
60 loss:  0.946017880161879


120 loss:  0.8791302449035784
180 loss:  0.8072804769731086


240 loss:  0.7308198162497399
300 loss:  0.6501930903037063


360 loss:  0.5659214427237933
420 loss:  0.47858725085533305


480 loss:  0.3888178090563346
540 loss:  0.29726857924696326


599 loss:  0.20615663572929388


## References

1. https://arxiv.org/pdf/2010.08561.pdf